# Bai Data Factory v2 — Kaggle T4×2 teacher pilot

GPT Work не нужен. Notebook использует уже pinned Qwen3-8B + DeepSeek teachers. Сначала 40 balanced calibration-кейсов; только при >=90% прохождении teacher/review contract запускаются остальные кейсы до полного pilot=400. Все результаты остаются review-only: никакого auto-Gold, retraining или release. Frozen promotion holdout здесь не используется.


In [ ]:
import sys, json, shutil, subprocess
from pathlib import Path
import torch
assert torch.cuda.device_count() >= 2, f'Нужны Kaggle T4 x2, получено: {torch.cuda.device_count()}'
!pip -q install -U 'transformers>=4.51,<5' accelerate bitsandbytes sentencepiece
!rm -rf /kaggle/working/tamdeshevle
!git clone --depth 1 --branch main https://github.com/eneonstudio-dev/tamdeshevle.git /kaggle/working/tamdeshevle
ROOT=Path('/kaggle/working/tamdeshevle')
sys.path.insert(0,str(ROOT/'teacher-lab/gpu'))
import kaggle_t4x2_teachers as teachers
import kaggle_factory_v2_runall as factory
print('GPU:',factory.verify_gpu(torch))
print('repo:',subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip())


In [ ]:
WORK=Path('/kaggle/working/bai_factory_v2'); WORK.mkdir(parents=True,exist_ok=True)
ALL=WORK/'all.jsonl'; PILOT=WORK/'pilot.jsonl'; CAL=WORK/'calibration.jsonl'
subprocess.check_call([sys.executable,str(ROOT/'teacher-lab/training/data_factory_v2.py'),str(ALL),'--pilot-out',str(PILOT)],cwd=ROOT)
tasks=[json.loads(x) for x in PILOT.read_text(encoding='utf-8').splitlines() if x.strip()]
assert len(tasks)==400 and len({x['id'] for x in tasks})==400
cal=factory.calibration_tasks(tasks,10)
CAL.write_text(''.join(json.dumps(x,ensure_ascii=False)+'\n' for x in cal),encoding='utf-8')
manifest={'schema_version':'2.0','kind':'bai_data_factory_v2_teacher_pilot','pilot_tasks':400,'calibration_tasks':40,'training_allowed':False,'requires_human_review':True,'frozen_promotion_holdout_used':False}
(WORK/'factory-manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
profiles=[json.loads((ROOT/'teacher-lab/profiles/deepseek-r1-distill-qwen-7b.json').read_text(encoding='utf-8')),json.loads((ROOT/'teacher-lab/profiles/qwen3-8b.json').read_text(encoding='utf-8'))]
corpus_manifest=factory.review_manifest(tasks,profiles)
(WORK/'corpus-manifest.json').write_text(json.dumps(corpus_manifest,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
print('Prepared:',len(tasks),'pilot /',len(cal),'calibration /',len(corpus_manifest['teachers']),'pinned teachers')


In [ ]:
RUN=WORK/'teacher_run'; RUN.mkdir(parents=True,exist_ok=True)
print('Loading pinned teachers once...')
loaded=teachers.load_teachers()
teachers.run_parallel(loaded,cal,RUN,700)
CAL_REVIEW=WORK/'reviews/calibration'
shutil.rmtree(CAL_REVIEW,ignore_errors=True); CAL_REVIEW.mkdir(parents=True)
subprocess.check_call(['node',str(ROOT/'teacher-lab/gpu/prepare_factory_review.mjs'),str(RUN),str(CAL),str(CAL_REVIEW)],cwd=ROOT)
cal_summary=json.loads((CAL_REVIEW/'summary.json').read_text(encoding='utf-8'))
print(json.dumps(cal_summary,ensure_ascii=False,indent=2))
factory.gate(cal_summary,40,0.90,'CALIBRATION')
print('CALIBRATION PASS — continuing to full 400')


In [ ]:
# Resume-aware: the 40 calibration task_ids are already done, so only the remaining 360 generate.
teachers.run_parallel(loaded,tasks,RUN,700)
FULL_REVIEW=WORK/'reviews/full'
shutil.rmtree(FULL_REVIEW,ignore_errors=True); FULL_REVIEW.mkdir(parents=True)
subprocess.check_call(['node',str(ROOT/'teacher-lab/gpu/prepare_factory_review.mjs'),str(RUN),str(PILOT),str(FULL_REVIEW)],cwd=ROOT)
full_summary=json.loads((FULL_REVIEW/'summary.json').read_text(encoding='utf-8'))
factory.gate(full_summary,400,1.0,'FULL')
subprocess.check_call(['node',str(ROOT/'teacher-lab/review-console.mjs'),str(FULL_REVIEW),str(FULL_REVIEW/'review.html')],cwd=ROOT)
TRIAGE=FULL_REVIEW/'triage'
subprocess.check_call([sys.executable,str(ROOT/'teacher-lab/training/summarize_factory_review.py'),'--tasks',str(PILOT),'--review-queue',str(FULL_REVIEW/'review-queue.json'),'--out-dir',str(TRIAGE)],cwd=ROOT)
shutil.copy2(RUN/'run-manifest.jsonl',FULL_REVIEW/'run-manifest.jsonl')
shutil.copy2(WORK/'corpus-manifest.json',FULL_REVIEW/'corpus-manifest.json')
review_zip=shutil.make_archive('/kaggle/working/bai_factory_v2_review','zip',FULL_REVIEW)
checkpoint=shutil.make_archive('/kaggle/working/bai_factory_v2_checkpoint','zip',WORK)
triage_summary=json.loads((TRIAGE/'review-triage-summary.json').read_text(encoding='utf-8'))
summary={'state':'REVIEW_READY','pilot_tasks':400,'calibration':cal_summary,'full':full_summary,'triage':triage_summary,'review_zip':review_zip,'checkpoint_zip':checkpoint,'training_allowed':False,'requires_human_review':True}
Path('/kaggle/working/bai_factory_v2_summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
print(json.dumps(summary,ensure_ascii=False,indent=2))
